# 📘 NotebookUtils (formerly MSSparkUtils) — Complete Training Guide
### Microsoft Fabric | Data Engineering | Spark Notebooks

---

**NotebookUtils** is a built-in utility package available in every Microsoft Fabric Spark Notebook. It provides a rich set of helpers for:

| Module | Purpose |
|---|---|
| `notebookutils.fs` | File system operations (ADLS Gen2, OneLake, local) |
| `notebookutils.notebook` | Notebook orchestration and chaining |
| `notebookutils.credentials` | Token retrieval and Azure Key Vault secrets |
| `notebookutils.variableLibrary` | Access centrally managed variable libraries |
| `notebookutils.lakehouse` | Lakehouse artifact management |
| `notebookutils.runtime` | Session context and metadata |
| `notebookutils.session` | Session lifecycle control |

> ⚠️ **Rename Notice:** `mssparkutils` has been officially renamed to `notebookutils`. The old namespace remains backward-compatible but will be retired in a future release. Always use `notebookutils`.

> ✅ **Runtime Requirement:** NotebookUtils requires **Spark 3.4 (Runtime v1.2) and above**.

## 📋 Table of Contents

1. [Getting Help](#1-getting-help)
2. [File System Utilities — `notebookutils.fs`](#2-file-system-utilities)
   - 2.1 List Files
   - 2.2 View File Properties
   - 2.3 Create Directory
   - 2.4 Copy File
   - 2.5 Fast Copy File
   - 2.6 Preview File Content
   - 2.7 Move File
   - 2.8 Write File
   - 2.9 Append to File
   - 2.10 Delete File or Directory
   - 2.11 Check File Existence
   - 2.12 Mount / Unmount Storage
3. [Notebook Utilities — `notebookutils.notebook`](#3-notebook-utilities)
   - 3.1 Run a Notebook
   - 3.2 Run Multiple Notebooks in Parallel (DAG)
   - 3.3 Exit a Notebook with a Value
   - 3.4 Manage Notebook Artifacts
4. [Credentials Utilities — `notebookutils.credentials`](#4-credentials-utilities)
   - 4.1 Get Token
   - 4.2 Get Secret from Key Vault
5. [Variable Library Utilities — `notebookutils.variableLibrary`](#5-variable-library-utilities)
6. [Lakehouse Utilities — `notebookutils.lakehouse`](#6-lakehouse-utilities)
7. [Runtime Utilities — `notebookutils.runtime`](#7-runtime-utilities)
8. [Session Utilities — `notebookutils.session`](#8-session-utilities)
9. [Quick Reference Cheat Sheet](#9-quick-reference-cheat-sheet)

---
## 1. Getting Help

Every module in NotebookUtils has a built-in `help()` method. This is the first thing to run when you want to explore available methods.

In [ ]:
# Top-level help — shows all available modules
notebookutils.help()

In [ ]:
# Help for a specific module
notebookutils.fs.help()

In [ ]:
# Help for a specific method within a module
notebookutils.fs.help("cp")

In [ ]:
# Help for the notebook module
notebookutils.notebook.help()

---
## 2. File System Utilities — `notebookutils.fs`

`notebookutils.fs` provides utilities to work with file systems including:
- **OneLake / ADLS Gen2** (`abfss://` paths)
- **Azure Blob Storage**
- **Local driver node file system** (`file:/` paths)
- **Default Lakehouse** (relative paths like `Files/...`)

### 📌 Path Reference

| Scenario | Example Path |
|---|---|
| Default Lakehouse (Spark notebook) | `Files/myfolder/myfile.csv` |
| ADLS Gen2 / OneLake absolute | `abfss://<container>@<account>.dfs.core.windows.net/<path>` |
| Local driver file system | `file:/tmp/myfile.txt` |

### 2.1 List Files — `notebookutils.fs.ls()`

Lists the contents of a directory. Returns an array of file info objects.

In [ ]:
# List files in the default Lakehouse 'Files' folder
notebookutils.fs.ls("Files/")

In [ ]:
# List files using absolute ABFSS path
notebookutils.fs.ls("abfss://<container_name>@<storage_account_name>.dfs.core.windows.net/<path>")

In [ ]:
# List files in the local driver file system
notebookutils.fs.ls("file:/tmp")

### 2.2 View File Properties

Each item returned by `ls()` has these properties:
- `name` — file or directory name
- `path` — full path
- `size` — size in bytes
- `isDir` — True if it's a directory
- `isFile` — True if it's a file

In [ ]:
# Iterate and print file properties
files = notebookutils.fs.ls("Files/")

for file in files:
    println(f"Name: {file.name}")
    println(f"  Path:   {file.path}")
    println(f"  Size:   {file.size} bytes")
    println(f"  isDir:  {file.isDir}")
    println(f"  isFile: {file.isFile}")

### 2.3 Create Directory — `notebookutils.fs.mkdirs()`

Creates the given directory if it does not exist. Also creates any necessary parent directories (like `mkdir -p` in Linux).

In [ ]:
# Create a folder in the default Lakehouse
notebookutils.fs.mkdirs("Files/training/output")

In [ ]:
# Create a folder using an absolute ABFSS path
notebookutils.fs.mkdirs("abfss://<container_name>@<storage_account_name>.dfs.core.windows.net/new_folder")

In [ ]:
# Create a folder in the local driver file system
notebookutils.fs.mkdirs("file:/tmp/my_local_dir")

### 2.4 Copy File — `notebookutils.fs.cp()`

Copies a file or directory, and supports copying **across file systems** (e.g., from ADLS to OneLake). Use `recurse=True` to copy entire directories.

In [ ]:
# Copy a single file within the default Lakehouse
notebookutils.fs.cp(
    "Files/raw/myfile.csv",        # source
    "Files/archive/myfile.csv"     # destination
)

In [ ]:
# Copy an entire directory recursively
notebookutils.fs.cp(
    "Files/raw/",
    "Files/backup/raw/",
    recurse=True    # copies all files and subdirectories
)

In [ ]:
# Copy across file systems (ADLS Gen2 to OneLake)
notebookutils.fs.cp(
    "abfss://source-container@source-account.dfs.core.windows.net/myfile.csv",
    "abfss://destination-container@onelake.dfs.fabric.microsoft.com/myfile.csv",
    recurse=False
)

### 2.5 Fast Copy File — `notebookutils.fs.fastcp()`

A higher-performance copy method that uses **AzCopy** under the hood. Recommended over `cp()` when dealing with **large data volumes**.

> ⚠️ `fastcp()` does **not** support copying files across OneLake regions. Use `cp()` for cross-region scenarios.

In [ ]:
# Fast copy — preferred for large datasets
notebookutils.fs.fastcp(
    "Files/large_dataset/",
    "Files/backup/large_dataset/",
    recurse=True
)

### 2.6 Preview File Content — `notebookutils.fs.head()`

Returns the first `maxBytes` bytes of a file as a UTF-8 string. Useful for quickly inspecting file content without fully reading it.

In [ ]:
# Preview the first 1024 bytes (default) of a file
content_preview = notebookutils.fs.head("Files/raw/myfile.csv")
print(content_preview)

In [ ]:
# Preview more bytes — e.g., first 5000 bytes
content_preview = notebookutils.fs.head("Files/raw/myfile.csv", 5000)
print(content_preview)

### 2.7 Move File — `notebookutils.fs.mv()`

Moves a file or directory. Supports moves across file systems. Parameters:
- **3rd param** (`createPath`): If `True`, creates parent directories if they don't exist
- **4th param** (`overwrite`): If `True`, overwrites the destination if it already exists

In [ ]:
# Move a file — creates parent directory if needed
notebookutils.fs.mv(
    "Files/raw/myfile.csv",         # source
    "Files/processed/myfile.csv",   # destination
    True                            # createPath = True
)

In [ ]:
# Move and overwrite at destination
notebookutils.fs.mv(
    "Files/raw/myfile.csv",
    "Files/processed/myfile.csv",
    True,   # createPath
    True    # overwrite
)

### 2.8 Write File — `notebookutils.fs.put()`

Writes a string to a file, encoded in UTF-8. If the file exists, set `overwrite=True` to replace it.

> ⚠️ Does not support concurrent writes to the same file.

In [ ]:
# Write a new file
notebookutils.fs.put(
    "Files/output/hello.txt",   # file path
    "Hello from Fabric!",       # content
    True                        # overwrite = True
)

### 2.9 Append to File — `notebookutils.fs.append()`

Appends a string to a file. If the file does not exist, set `createFileIfNotExists=True` to create it.

> ⚠️ When calling `append()` in a loop on the same file, add a **0.5–1 second sleep** between writes. The internal `flush` is asynchronous, and rapid writes can cause data integrity issues.

In [ ]:
# Append a line to an existing log file
notebookutils.fs.append(
    "Files/logs/pipeline.log",          # file path
    "2024-01-15 10:30:00 | INFO | Pipeline started\n",  # content
    True                                 # createFileIfNotExists
)

In [ ]:
# Appending in a loop — always add a sleep to ensure data integrity
import time

log_entries = [
    "Step 1: Ingestion complete\n",
    "Step 2: Transformation complete\n",
    "Step 3: Load complete\n"
]

for entry in log_entries:
    notebookutils.fs.append("Files/logs/pipeline.log", entry, True)
    time.sleep(0.5)   # ← Required when appending in a loop

### 2.10 Delete File or Directory — `notebookutils.fs.rm()`

Removes a file or directory. Use `recurse=True` to delete a directory and all of its contents.

In [ ]:
# Delete a single file
notebookutils.fs.rm("Files/temp/myfile.csv")

In [ ]:
# Delete an entire directory and all its contents
notebookutils.fs.rm(
    "Files/temp/",
    recurse=True    # required to delete non-empty directories
)

### 2.11 Check File Existence — `notebookutils.fs.exists()`

Returns `True` if the file or directory exists, `False` otherwise. Very useful for conditional logic in pipelines.

In [ ]:
# Check if a file exists
file_path = "Files/raw/myfile.csv"

if notebookutils.fs.exists(file_path):
    print(f"✅ File found: {file_path}")
else:
    print(f"❌ File not found: {file_path}")

In [ ]:
# Practical pattern: create output directory only if it doesn't exist
output_dir = "Files/processed/2024/01/"

if not notebookutils.fs.exists(output_dir):
    notebookutils.fs.mkdirs(output_dir)
    print(f"📁 Created directory: {output_dir}")
else:
    print(f"📁 Directory already exists: {output_dir}")

### 2.12 Mount / Unmount Storage — `notebookutils.fs.mount()`

Mounts a remote storage directory (ADLS Gen2, Blob Storage, or a Lakehouse) to a local mount point. After mounting, you can access files using Spark or local paths.

**Authentication options:** `accountKey` or `sasToken` (store in Azure Key Vault — never hardcode!)

**Key parameters:**
- `fileCacheTimeout` (default: 120s) — how long files are cached locally before re-fetching from the server
- `timeout` (default: 120s) — mount operation timeout

> ⚠️ Mount points are **job-level** (not persisted between sessions). Always call `unmount()` explicitly at the end of your notebook to release resources.

In [ ]:
# Mount using Account Key (retrieved from Key Vault — never hardcode the key!)
accountKey = notebookutils.credentials.getSecret(
    "https://<your-keyvault>.vault.azure.net/",
    "<secret-name>"
)

notebookutils.fs.mount(
    "abfss://mycontainer@<accountname>.dfs.core.windows.net",  # remote storage
    "/mymount",                                                 # local mount point
    {"accountKey": accountKey}
)

In [ ]:
# Mount using SAS Token
sasToken = notebookutils.credentials.getSecret(
    "https://<your-keyvault>.vault.azure.net/",
    "<sas-secret-name>"
)

notebookutils.fs.mount(
    "abfss://mycontainer@<accountname>.dfs.core.windows.net",
    "/mymount",
    {"sasToken": sasToken}
)

In [ ]:
# Mount a Fabric Lakehouse directly
notebookutils.fs.mount(
    "abfss://<workspace_name>@onelake.dfs.fabric.microsoft.com/<lakehouse_name>.Lakehouse",
    "/mylakehouse"
)

In [ ]:
# Mount with custom cache timeout (set to 0 to always fetch latest file)
notebookutils.fs.mount(
    "abfss://mycontainer@<accountname>.dfs.core.windows.net",
    "/freshdata",
    {
        "accountKey": accountKey,
        "fileCacheTimeout": 0,   # always get the latest version
        "timeout": 200           # increase if mount times out on large clusters
    }
)

In [ ]:
# Access files under the mount point
# Use getMountPath() to get the exact local path
mount_path = notebookutils.fs.getMountPath("/mymount")
print(f"Local mount path: {mount_path}")

# List files via notebookutils.fs
notebookutils.fs.ls(f"file://{mount_path}")

# Or read with Spark
spark.read.csv(f"file://{mount_path}/myfile.csv", header=True).show()

In [ ]:
# List all current mount points
notebookutils.fs.mounts()

In [ ]:
# Unmount when done — always do this explicitly at the end of your notebook
notebookutils.fs.unmount("/mymount")
print("✅ Unmounted /mymount")

---
## 3. Notebook Utilities — `notebookutils.notebook`

These utilities let you **chain notebooks together**, pass parameters between them, run them in parallel, and manage notebook artifacts programmatically.

### 3.1 Run a Notebook — `notebookutils.notebook.run()`

Runs a child notebook and returns its exit value. The child notebook runs on the same Spark pool as the parent.

**Signature:** `run(path, timeoutSeconds, arguments, workspaceId)`

> ℹ️ You can view a **snapshot link** in cell output to inspect the child notebook's run results.

In [ ]:
# Run a notebook with default parameters
exit_value = notebookutils.notebook.run(
    "nb_transform_data",   # notebook name (same workspace)
    90                     # timeout in seconds
)
print(f"Child notebook returned: {exit_value}")

In [ ]:
# Run a notebook and pass parameters
exit_value = notebookutils.notebook.run(
    "nb_process_file",
    120,                                    # timeout
    {"file_date": "2024-01-15",
     "client_code": "TUFTS",
     "max_retries": 3}
)
print(f"Result: {exit_value}")

In [ ]:
# Run a notebook from a DIFFERENT workspace (requires Runtime v1.2+)
exit_value = notebookutils.notebook.run(
    "nb_shared_utility",
    90,
    {"input": "value"},
    "fe0a6e2a-a909-4aa3-a698-0a651de790aa"   # workspace ID
)
print(f"Result: {exit_value}")

### 3.2 Run Multiple Notebooks in Parallel — `notebookutils.notebook.runMultiple()`

Runs multiple notebooks simultaneously, or in a predefined dependency order (DAG — Directed Acyclic Graph). All notebooks share compute resources within the same Spark session.

**DAG configuration options:**
- `name` — unique activity name
- `path` — notebook name
- `args` — parameters to pass
- `dependencies` — list of activity names that must complete first
- `retry` — number of retries on failure
- `retryIntervalInSeconds` — wait between retries
- `timeoutPerCellInSeconds` — per-cell timeout (default: 90s)
- `timeoutInSeconds` — overall DAG timeout (default: 12 hours)
- `concurrency` — max notebooks running at once (default: 50 for Spark)

In [ ]:
# Simple parallel run — all notebooks start at the same time
notebookutils.notebook.runMultiple([
    "nb_process_client_A",
    "nb_process_client_B",
    "nb_process_client_C"
])

In [ ]:
# Advanced DAG orchestration with dependencies
# In this example:
#   - Ingest_A and Ingest_B run in parallel
#   - Transform runs only after BOTH ingestion steps complete

DAG = {
    "activities": [
        {
            "name": "Ingest_A",
            "path": "nb_ingest_source_a",
            "timeoutPerCellInSeconds": 120,
            "args": {"source": "ADLS", "date": "2024-01-15"}
        },
        {
            "name": "Ingest_B",
            "path": "nb_ingest_source_b",
            "timeoutPerCellInSeconds": 120,
            "args": {"source": "SFTP", "date": "2024-01-15"}
        },
        {
            "name": "Transform",
            "path": "nb_transform_and_merge",
            "timeoutPerCellInSeconds": 300,
            "args": {"date": "2024-01-15"},
            "retry": 2,
            "retryIntervalInSeconds": 30,
            "dependencies": ["Ingest_A", "Ingest_B"]  # waits for both to finish
        }
    ],
    "timeoutInSeconds": 3600,   # 1 hour total timeout
    "concurrency": 10
}

results = notebookutils.notebook.runMultiple(DAG)

# Print exit values from each activity
for name, result in results.items():
    print(f"{name}: {result.exitValue}")

In [ ]:
# Validate your DAG definition before running
is_valid = notebookutils.notebook.validateDAG(DAG)
print(f"DAG is valid: {is_valid}")

### 3.3 Exit a Notebook — `notebookutils.notebook.exit()`

Exits the current notebook and passes a return value to the caller (parent notebook or pipeline).

**Behavior varies by context:**
- **Interactive run:** Throws an exception and stops subsequent cells (Spark session stays alive)
- **Pipeline run:** Notebook activity completes with the exit value; Spark session stops
- **Reference run (called via `run()`):** Stops the child notebook; parent continues from the next cell

> ⚠️ Always call `exit()` in a **dedicated cell** — it overwrites the current cell's output.

In [ ]:
# Exit with a simple string value
notebookutils.notebook.exit("SUCCESS")

In [ ]:
# Exit with a JSON payload — useful for passing structured results to a pipeline
import json

result = {
    "status": "SUCCESS",
    "rows_processed": 1500,
    "files_ingested": 5,
    "timestamp": "2024-01-15T10:30:00Z"
}

notebookutils.notebook.exit(json.dumps(result))

In [ ]:
# Typical pattern: parent notebook reads the exit value from child
exit_raw = notebookutils.notebook.run("nb_child_notebook", 120, {"date": "2024-01-15"})
exit_data = json.loads(exit_raw)

print(f"Status:          {exit_data['status']}")
print(f"Rows processed:  {exit_data['rows_processed']}")
print(f"Files ingested:  {exit_data['files_ingested']}")

### 3.4 Manage Notebook Artifacts

You can programmatically **create, read, update, delete, and list** notebook items in a workspace.

In [ ]:
# Create a new notebook artifact from content
notebookutils.notebook.create(
    "nb_generated_notebook",           # name
    'spark.sql("SELECT 1 as col1")',  # content
    "Spark"                            # language
)

In [ ]:
# Read a notebook's content
notebook_content = notebookutils.notebook.read("nb_my_notebook")
print(notebook_content)

In [ ]:
# Update an existing notebook
notebookutils.notebook.update(
    "nb_generated_notebook",
    'spark.sql("SELECT 2 as col1")'   # new content
)

In [ ]:
# Delete a notebook
notebookutils.notebook.delete("nb_generated_notebook")

In [ ]:
# List all notebooks in the workspace
notebooks = notebookutils.notebook.list()
for nb in notebooks:
    print(f"Notebook: {nb}")

---
## 4. Credentials Utilities — `notebookutils.credentials`

Securely retrieve credentials (tokens, secrets) without hardcoding them in your notebook.

### 4.1 Get Token — `notebookutils.credentials.getToken()`

Retrieves an OAuth2 token for a specified resource (e.g., Azure Data Lake, SQL Database, Key Vault).

In [ ]:
# Get token for Azure Storage (dfs.core.windows.net)
token = notebookutils.credentials.getToken("https://dfs.core.windows.net")
print(f"Token acquired: {token[:50]}...")

In [ ]:
# Get token for Azure SQL Database
token = notebookutils.credentials.getToken("https://database.windows.net")
print(f"SQL Token acquired")

### 4.2 Get Secret from Key Vault — `notebookutils.credentials.getSecret()`

Retrieves a secret stored in Azure Key Vault. The vault URL and secret name are parameters.

In [ ]:
# Retrieve a secret from Key Vault
secret_value = notebookutils.credentials.getSecret(
    "https://<your-keyvault>.vault.azure.net/",
    "<secret-name>"
)
print(f"Secret retrieved: {secret_value[:20]}...")

In [ ]:
# Practical example: use Key Vault secret to connect to a database
db_password = notebookutils.credentials.getSecret(
    "https://my-keyvault.vault.azure.net/",
    "db-password"
)

# Use password in connection string (pseudocode)
connection_string = f"Server=myserver.database.windows.net;User=sa;Password={db_password};Database=mydb"
print("Connection string configured")

---
## 5. Variable Library Utilities — `notebookutils.variableLibrary`

Access and manage centrally managed variable libraries in Fabric.

In [ ]:
# Get a variable from a variable library
variable_value = notebookutils.variableLibrary.get("my_library", "variable_name")
print(f"Variable value: {variable_value}")

In [ ]:
# Set a variable in a variable library
notebookutils.variableLibrary.set("my_library", "variable_name", "new_value")
print("Variable updated")

---
## 6. Lakehouse Utilities — `notebookutils.lakehouse`

Programmatically manage Lakehouse artifacts (tables, shortcuts, etc.).

In [ ]:
# List tables in the default Lakehouse
tables = notebookutils.lakehouse.list()
for table in tables:
    print(f"Table: {table}")

In [ ]:
# Delete a Lakehouse table
notebookutils.lakehouse.delete(
    "my_table",    # table name
    "abfss://..."  # table path (optional)
)

---
## 7. Runtime Utilities — `notebookutils.runtime`

Get session and notebook context information.

In [ ]:
# Get notebook context
context = notebookutils.runtime.getContext()
print(f"Workspace ID: {context['workspaceId']}")
print(f"Notebook Name: {context['notebookName']}")
print(f"User ID: {context['userId']}")

---
## 8. Session Utilities — `notebookutils.session`

Control the Spark session lifecycle.

In [ ]:
# Configure session to use batch mode (improves performance for certain workloads)
notebookutils.session.useBatch()

---
## 9. Quick Reference Cheat Sheet

### File System
```scala
notebookutils.fs.ls(path)                    # List directory
notebookutils.fs.mkdirs(path)                # Create directory
notebookutils.fs.cp(source, dest, recurse)   # Copy files
notebookutils.fs.mv(source, dest, create, overwrite)  # Move files
notebookutils.fs.rm(path, recurse)           # Delete
notebookutils.fs.head(path, maxBytes)        # Preview file
notebookutils.fs.put(path, content, overwrite)  # Write file
notebookutils.fs.append(path, content, create)  # Append to file
notebookutils.fs.exists(path)                # Check if exists
notebookutils.fs.mount(source, mount, config)  # Mount storage
notebookutils.fs.unmount(mount)              # Unmount
```

### Notebooks
```scala
notebookutils.notebook.run(path, timeout, args)  # Run notebook
notebookutils.notebook.runMultiple(list/DAG)     # Run in parallel
notebookutils.notebook.exit(value)               # Exit with value
notebookutils.notebook.create(name, content, lang)  # Create notebook
notebookutils.notebook.read(name)                # Read notebook
notebookutils.notebook.update(name, content)     # Update notebook
notebookutils.notebook.delete(name)              # Delete notebook
notebookutils.notebook.list()                    # List notebooks
```

### Credentials
```scala
notebookutils.credentials.getToken(resource)    # Get OAuth2 token
notebookutils.credentials.getSecret(vault, name)  # Get Key Vault secret
```

### Variables
```scala
notebookutils.variableLibrary.get(library, name)    # Get variable
notebookutils.variableLibrary.set(library, name, value)  # Set variable
```

### Lakehouse
```scala
notebookutils.lakehouse.list()       # List tables
notebookutils.lakehouse.delete(name) # Delete table
```

### Runtime & Session
```scala
notebookutils.runtime.getContext()   # Get notebook context
notebookutils.session.useBatch()     # Use batch mode
```